# 🔬 Demo: Clasificador de eventos Super-Kamiokande
## Usando el modelo CNN pre-entrenado en imágenes del monitor en tiempo real

---

## ¿Qué hace este notebook?

En el taller anterior **entrenamos** una Red Neuronal Convolucional (CNN) desde cero para distinguir dos tipos de eventos Cherenkov en el detector Super-Kamiokande:

| Clase | Partícula | Aspecto del anillo |
|-------|-----------|-------------------|
| `mu_like` | Muón (µ) | Borde **nítido y continuo** |
| `e_like`  | Electrón / positrón | Borde **difuso o granulado** |

Este notebook es diferente: **no entrenamos nada**. Cargamos directamente el modelo ya entrenado y lo usamos para clasificar imágenes nuevas del monitor en tiempo real.

> 🔗 **Monitor en tiempo real de Super-Kamiokande:**  
> **[https://www-sk.icrr.u-tokyo.ac.jp/realtimemonitor/](https://www-sk.icrr.u-tokyo.ac.jp/realtimemonitor/)**  
> Tenlo abierto en otra pestaña — lo usarás en la Sección 4.

---

## ¿Por qué es útil cargar un modelo pre-entrenado?

Entrenar una CNN desde cero puede tomar horas o días con datasets grandes y requiere potencia computacional considerable. Una vez entrenado, el modelo puede **guardarse** y **reutilizarse** indefinidamente:

- Sin re-entrenar
- En segundos
- Por cualquier persona con acceso al archivo

Esto es exactamente lo que hacen los equipos de física en experimentos reales: un grupo entrena el modelo, lo valida científicamente, y luego toda la colaboración lo usa para analizar datos nuevos.

---

## Mapa de este notebook

```
Sección 1 → Importar librerías
Sección 2 → Montar Drive y cargar el modelo pre-entrenado
Sección 3 → Entender el modelo: resumen de arquitectura
Sección 4 → Capturar imagen del monitor SK y subirla
Sección 5 → Preprocesar la imagen (igual que en el entrenamiento)
Sección 6 → Clasificar el evento y visualizar el resultado
Sección 7 → (Opcional) Clasificar múltiples imágenes de una vez
```

> 💡 **Cómo ejecutar:** `Shift + Enter` en cada celda, de arriba hacia abajo.


---
## 🛠️ Sección 1 — Importar librerías

### ¿Qué necesitamos para usar un modelo ya entrenado?

Mucho menos que para entrenarlo. Solo necesitamos:

| Librería | Para qué |
|----------|----------|
| **TensorFlow / Keras** | Cargar y ejecutar el modelo CNN |
| **NumPy** | Manipular los arreglos de píxeles |
| **Matplotlib** | Visualizar la imagen y el resultado |
| **PIL (Pillow)** | Leer y redimensionar la imagen subida |

> ✅ **Resultado esperado:** Las versiones de TensorFlow y la GPU disponible.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image, UnidentifiedImageError
import tensorflow as tf
import os

print("TensorFlow:", tf.__version__)
print("GPU disponible:", tf.config.list_physical_devices('GPU'))
print("\n✅ Librerías listas.")


---
## 📂 Sección 2 — Montar Drive y cargar el modelo pre-entrenado

### ¿Dónde está el modelo?

Al final del taller de entrenamiento guardamos el modelo en:
```
Mi unidad / colab_shared / model_mu_e.keras
```

El archivo `.keras` contiene todo lo que la red aprendió:
- La **arquitectura** (cuántas capas, de qué tipo)
- Los **pesos** (los millones de números que representan el conocimiento adquirido)
- La **configuración de compilación** (optimizador, función de pérdida)

**Analogía:** Es como guardar el cerebro de la red en un disco duro. Al cargarlo, la red despierta exactamente donde la dejamos — no necesita volver a estudiar.

> ⚠️ **Antes de ejecutar:** Asegúrate de que `model_mu_e.keras` esté en la carpeta `colab_shared` de tu Google Drive.  
> ✅ **Resultado esperado:** `Modelo cargado correctamente ✓`


In [ ]:
from google.colab import drive

# Montar Google Drive
drive.mount('/content/drive')

# Ruta donde guardamos el modelo en el taller de entrenamiento
MODEL_PATH = '/content/drive/MyDrive/colab_shared/model_mu_e.keras'

if not os.path.exists(MODEL_PATH):
    raise FileNotFoundError(
        f"No encontré el modelo en: {MODEL_PATH}\n"
        "Verifica que el archivo esté en tu Google Drive en la carpeta colab_shared."
    )

model = tf.keras.models.load_model(MODEL_PATH)
print(f"Modelo cargado correctamente ✓")
print(f"Ruta: {MODEL_PATH}")


---
## 🧠 Sección 3 — Conocer el modelo: arquitectura y parámetros

### ¿Qué hay dentro del modelo que acabamos de cargar?

Podemos inspeccionarlo como si abriéramos el capó de un motor. `model.summary()` muestra:

- Cada **capa** de la red y su tipo
- El **tamaño de los datos** que pasan por cada capa (Output Shape)
- La cantidad de **parámetros** (pesos entrenables) en cada capa

### ¿Qué esperar?

La arquitectura es la CNN que construimos en el taller:
```
Conv2D(32) → MaxPool → Conv2D(64) → MaxPool → Conv2D(128) → MaxPool
→ Flatten → Dense(128) → Dropout(0.5) → Dense(1, sigmoid)
```

La salida final es **un solo número entre 0 y 1**:
- `> 0.5` → la red predice `MU-like` (muón)
- `≤ 0.5` → la red predice `E-like` (electrón/positrón)

> 🤔 **Observa:** ¿Cuántos parámetros tiene en total? Recuerda que cada uno de esos números fue ajustado durante el entrenamiento mirando ejemplos reales del detector Super-K.


In [ ]:
# Mostrar la arquitectura completa del modelo cargado
model.summary()

# Datos de entrada y salida esperados
input_shape  = model.input_shape   # (None, 224, 224, 1)
output_shape = model.output_shape  # (None, 1)
print(f"\nEntrada esperada: {input_shape}  →  imagen de 224×224 píxeles en escala de grises")
print(f"Salida:           {output_shape}  →  un número en [0, 1]  (probabilidad de ser mu_like)")


---
## 📸 Sección 4 — Capturar y subir una imagen del monitor de Super-K

### ¡La parte más emocionante!

Vas a capturar un evento **real** del detector Super-Kamiokande en este momento y dejarlo que tu modelo lo clasifique.

### Instrucciones paso a paso:

**1. Abre el monitor:**  
👉 [https://www-sk.icrr.u-tokyo.ac.jp/realtimemonitor/](https://www-sk.icrr.u-tokyo.ac.jp/realtimemonitor/)

**2. Identifica el "barrel" (panel central):**  
Es la franja horizontal del medio — las paredes cilíndricas del detector desenrolladas en 2D. Los anillos Cherenkov son más visibles ahí. Los paneles de arriba y abajo son las tapas (top/bottom cap).

**3. Observa el anillo:**  
- ¿El borde se ve **nítido y definido**? → probablemente `mu_like`
- ¿El borde se ve **borroso o granulado**? → probablemente `e_like`
- Haz tu hipótesis antes de que el modelo clasifique.

**4. Toma captura de pantalla** (`Ctrl+Shift+S` o la herramienta de recorte de tu OS) y recórtala dejando solo el panel barrel.

**5. Ejecuta la celda de abajo** → aparecerá un botón para subir tu imagen (PNG o JPG).

> 💡 **Tip:** Puedes subir cualquier imagen para probar, incluso una que no sea del detector. ¿Qué crees que pasará? ¿El modelo dirá algo con sentido o una respuesta aleatoria? Eso nos habla de los **límites de la IA**.


In [ ]:
from google.colab import files

print("Selecciona la imagen del barrel de Super-K cuando aparezca el botón...")
uploaded = files.upload()

if not uploaded:
    raise RuntimeError("No se subió ningún archivo. Vuelve a ejecutar esta celda y selecciona una imagen.")

IMG_FNAME = list(uploaded.keys())[0]
print(f"\nArchivo recibido: {IMG_FNAME} ✓")


---
## ⚙️ Sección 5 — Preprocesar la imagen

### ¿Por qué necesitamos preprocesar?

La red neuronal fue entrenada con imágenes en un formato muy específico. Para que el modelo funcione correctamente, **cualquier imagen nueva debe tener exactamente el mismo formato** que las imágenes de entrenamiento. Si no, es como darle a alguien un examen en un idioma diferente al que estudió.

### Transformaciones que aplicamos:

| Paso | Descripción | ¿Por qué? |
|------|-------------|-----------|
| **Escala de grises** | Convertir a 1 canal de color | El modelo fue entrenado en imágenes grises (1 canal), no RGB (3 canales) |
| **Redimensionar a 224×224** | Cambiar el tamaño | La CNN espera exactamente esta resolución de entrada |
| **Normalizar a [0, 1]** | Dividir cada píxel entre 255 | Los pesos de la red se ajustaron esperando valores en este rango |
| **Agregar dimensiones** | De (224,224) a (1,224,224,1) | Keras espera un *lote* de imágenes con canal explícito: `(batch, alto, ancho, canales)` |

> ✅ **Resultado esperado:** Se muestra la imagen procesada tal como la ve la red (en escala de grises, 224×224).


In [ ]:
def preprocesar_imagen(path):
    """
    Carga una imagen y la convierte al formato exacto que espera el modelo:
    - Escala de grises (1 canal)
    - 224 × 224 píxeles
    - Valores normalizados entre 0 y 1
    - Shape: (1, 224, 224, 1)  ← batch de 1 imagen
    """
    try:
        img = Image.open(path).convert('L')          # 'L' = escala de grises
    except UnidentifiedImageError:
        import cv2
        arr = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
        if arr is None:
            raise ValueError("No pude leer la imagen. Usa formato PNG o JPG.")
        img = Image.fromarray(arr)

    img  = img.resize((224, 224))                    # redimensionar
    arr  = np.asarray(img).astype('float32') / 255.0 # normalizar a [0, 1]
    arr  = np.expand_dims(arr, axis=(0, -1))         # (1, 224, 224, 1)
    return arr, img

# Preprocesar la imagen subida
x_input, img_pil = preprocesar_imagen(IMG_FNAME)

print(f"Shape del tensor de entrada: {x_input.shape}")
print(f"Valor mínimo de píxel: {x_input.min():.3f}  |  máximo: {x_input.max():.3f}")
print(f"\nAsí es como la red 've' tu imagen:")

plt.figure(figsize=(5, 4))
plt.imshow(x_input[0, :, :, 0], cmap='gray')
plt.title("Imagen preprocesada (224×224, escala de grises)")
plt.axis('off')
plt.tight_layout()
plt.show()


---
## 🤖 Sección 6 — Clasificar el evento y ver el resultado

### ¿Qué hace `model.predict()`?

Con el modelo cargado y la imagen preprocesada, el paso final es la **inferencia**: pasar la imagen hacia adelante por todas las capas de la red hasta obtener la predicción.

A diferencia del entrenamiento (que modifica los pesos), la inferencia es solo una **consulta**: los pesos permanecen fijos, la red simplemente aplica lo que aprendió.

### ¿Cómo interpretar la salida?

La última capa del modelo usa la función **Sigmoid**, que comprime cualquier valor a un rango entre 0 y 1. Ese número es la **probabilidad de que el evento sea mu_like**:

```
p(mu_like) = 0.95  →  95% muón     →  MU-like  ✓
p(mu_like) = 0.03  →  97% electrón →  E-like   ✓
p(mu_like) = 0.51  →  51% muón     →  MU-like  (pero con poca confianza)
```

> 🤔 **Reflexión:** ¿Cuánta confianza tiene tu modelo? ¿Coincide con tu hipótesis visual de la Sección 4?


In [ ]:
# Warm-up de la GPU (inicializa cuDNN si es la primera inferencia)
_ = model.predict(np.zeros((1, 224, 224, 1), dtype='float32'), verbose=0)

# ── Inferencia ──────────────────────────────────────────────────────────────
prob_mu = float(model.predict(x_input, verbose=0)[0][0])
prob_e  = 1.0 - prob_mu
prediccion = "MU-like  (muón µ)" if prob_mu > 0.5 else "E-like  (electrón / positrón)"
confianza  = max(prob_mu, prob_e) * 100
color_bar  = 'steelblue' if prob_mu > 0.5 else 'tomato'

# ── Visualización ────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Panel izquierdo: imagen
axes[0].imshow(x_input[0, :, :, 0], cmap='gray')
axes[0].set_title(f"Evento Super-K\nArchivo: {IMG_FNAME}", fontsize=11)
axes[0].axis('off')

# Panel derecho: probabilidades
clases = ['E-like\n(electrón)', 'MU-like\n(muón)']
probs  = [prob_e, prob_mu]
bars   = axes[1].barh(clases, probs, color=['tomato', 'steelblue'], edgecolor='white', height=0.5)
axes[1].set_xlim(0, 1)
axes[1].axvline(0.5, color='gray', linestyle='--', linewidth=1, label='Umbral 50%')
axes[1].set_xlabel("Probabilidad", fontsize=11)
axes[1].set_title("Predicción del modelo CNN", fontsize=11)
axes[1].legend(fontsize=9)
for bar, prob in zip(bars, probs):
    axes[1].text(bar.get_width() + 0.02, bar.get_y() + bar.get_height()/2,
                 f"{prob*100:.1f}%", va='center', fontsize=12, fontweight='bold')

plt.suptitle(f"Predicción: {prediccion}  |  Confianza: {confianza:.1f}%",
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print(f"\n{'='*50}")
print(f"  Predicción : {prediccion}")
print(f"  p(mu_like) : {prob_mu:.4f}   ({prob_mu*100:.1f}%)")
print(f"  p(e_like)  : {prob_e:.4f}   ({prob_e*100:.1f}%)")
print(f"  Confianza  : {confianza:.1f}%")
print(f"{'='*50}")


---
## 📦 Sección 7 (Opcional) — Clasificar varias imágenes de una sola vez

### Inferencia en lote (batch inference)

En un experimento real, Super-Kamiokande registra miles de eventos por día. No es práctico clasificarlos uno por uno — se procesan en **lotes**.

Esta sección te permite subir varias capturas del monitor y obtener la predicción para todas de una vez, con un resumen estadístico al final.

> 💡 Sube entre 2 y 10 imágenes del barrel (distintos eventos) para ver cómo varía la predicción entre eventos.


In [ ]:
from google.colab import files as colab_files

print("Selecciona varias imágenes del barrel de Super-K...")
uploaded_batch = colab_files.upload()

if not uploaded_batch:
    raise RuntimeError("No se subió ningún archivo.")

resultados = []
nombres    = list(uploaded_batch.keys())

print(f"\nClasificando {len(nombres)} imagen(es)...\n")

fig, axes = plt.subplots(1, len(nombres), figsize=(5 * len(nombres), 4))
if len(nombres) == 1:
    axes = [axes]

for ax, fname in zip(axes, nombres):
    try:
        x, _ = preprocesar_imagen(fname)
        p_mu  = float(model.predict(x, verbose=0)[0][0])
        p_e   = 1.0 - p_mu
        pred  = "MU-like" if p_mu > 0.5 else "E-like"
        conf  = max(p_mu, p_e) * 100
        color = 'steelblue' if p_mu > 0.5 else 'tomato'

        ax.imshow(x[0, :, :, 0], cmap='gray')
        ax.set_title(f"{pred}\nConfianza: {conf:.1f}%\n{fname[:20]}", fontsize=9, color=color)
        ax.axis('off')

        resultados.append({
            "archivo": fname,
            "prediccion": pred,
            "p_mu": p_mu,
            "p_e": p_e,
            "confianza": conf
        })
    except Exception as ex:
        ax.set_title(f"Error\n{fname[:20]}", fontsize=9, color='gray')
        ax.axis('off')
        print(f"  ⚠️  No pude procesar '{fname}': {ex}")

plt.suptitle("Clasificación por lote — Eventos Super-K", fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

# Resumen
print("\n" + "="*55)
print(f"{'Archivo':<25} {'Predicción':<12} {'Confianza':>10}")
print("-"*55)
for r in resultados:
    print(f"{r['archivo'][:24]:<25} {r['prediccion']:<12} {r['confianza']:>9.1f}%")
print("="*55)
n_mu = sum(1 for r in resultados if r['prediccion'] == "MU-like")
n_e  = len(resultados) - n_mu
print(f"\nResumen:  MU-like → {n_mu}   |   E-like → {n_e}   |   Total → {len(resultados)}")


---
## 🎓 ¿Qué aprendiste en este notebook?

| Concepto | Lo que hiciste |
|----------|---------------|
| **Cargar modelo pre-entrenado** | Importaste el archivo `.keras` desde Drive sin re-entrenar |
| **Inspeccionar arquitectura** | Usaste `model.summary()` para ver las capas y parámetros |
| **Preprocesamiento consistente** | Aplicaste el mismo pipeline que en el entrenamiento (gris, 224×224, normalización) |
| **Inferencia** | Corriste `model.predict()` — solo lectura, sin modificar los pesos |
| **Interpretación probabilística** | Entendiste que la salida `p(mu_like)` es una probabilidad, no solo 0 ó 1 |
| **Batch inference** | Clasificaste múltiples imágenes de una sola vez |

### La conexión con el mundo real

En Super-Kamiokande y experimentos similares, el pipeline completo se ve así:

```
Detector registra evento
        ↓
Sistema de adquisición de datos guarda la imagen
        ↓
Pipeline automático preprocesa (igual que nuestra Sección 5)
        ↓
Modelo CNN clasifica (igual que nuestra Sección 6)
        ↓
El resultado se almacena en la base de datos de la colaboración
        ↓
Físicos analizan la distribución estadística de miles de eventos
        ↓
Publicación científica con resultados sobre neutrinos
```

Lo que hoy hiciste en un notebook de Colab es, en esencia, lo mismo que hacen los sistemas de análisis en tiempo real de los grandes experimentos de física de partículas.

---

> *La diferencia entre un juguete académico y una herramienta científica real no es el principio — es la escala y la validación rigurosa de los resultados.*
